# ScenarioKnowledgeGraph Explorer

Interactive notebook for scenario exploration, retrieval, and benchmark analysis.

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from skg.config import get_settings
from skg.retrieval.embed import embed_scenarios
from skg.retrieval.dense import DenseIndex
from skg.retrieval.sparse import SparseIndex
from skg.retrieval.engine import HybridRetriever
from skg.graph import MemoryGraphStore
from skg.graph.loader import load_graph_from_parquet

settings = get_settings()
print(f"Data directory: {settings.raw_data_dir}")

## Load Data

In [ ]:
# Load scenario summary
summary_df = pd.read_parquet(settings.raw_data_dir / "scenario_summary.parquet")
print(f"Loaded {len(summary_df)} scenarios")
print(summary_df.head())

In [ ]:
# Load graph
graph_store = MemoryGraphStore()
graph_stats = load_graph_from_parquet(graph_store, settings.graph_data_dir)
print(f"Graph: {graph_stats['n_nodes_loaded']} nodes, {graph_stats['n_edges_loaded']} edges")

## Build Retrieval Engine

In [ ]:
# Create embeddings and indexes
embeddings = embed_scenarios(summary_df)
dense = DenseIndex(embeddings)
sparse = SparseIndex(summary_df["description"].tolist())
retriever = HybridRetriever(summary_df, dense, sparse, graph_store)
print("Retriever ready")

## Scenario Search

In [ ]:
# Search for scenarios
query = "cut-in with TTC"
results = retriever.search(query, k=10)

results_df = pd.DataFrame(results)
print(f"Query: {query}")
print(results_df[['scenario_id', 'behavior_label', 'score']])

## Behavior Distribution

In [ ]:
# Plot behavior distribution
behavior_counts = summary_df['behavior_label'].value_counts()
print(behavior_counts)
behavior_counts.plot(kind='bar')

## Kinematic Statistics

In [ ]:
# TTC and speed distributions
print(summary_df[['min_ttc', 'max_speed']].describe())
summary_df[['min_ttc', 'max_speed']].hist()

## Benchmark Results

In [ ]:
# Load benchmark results
if (settings.results_dir / 'benchmark.json').exists():
    with open(settings.results_dir / 'benchmark.json') as f:
        bench = json.load(f)
    
    print("Benchmark Summary:")
    print(f"  Scenarios: {bench.get('n_scenarios')}")
    print(f"  Nodes: {bench.get('n_nodes')}")
    print(f"  Edges: {bench.get('n_edges')}")
    
    if 'latencies' in bench:
        lat_df = pd.DataFrame([{'query': q, **v} for q, v in bench['latencies'].items()])
        print("\nLatencies (ms):")
        print(lat_df)